<div align="center">

# Week 02: Continuous Dynamics: Batteries, Buildings and Grid Frequency

**Cyber-Physical Systems (BIL-435), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-02) · [Lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lecture.html) · [Interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lab.html) · [PDF notes](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/Week02_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also stores the constants of the three models of this week. They describe the battery, the library and the island. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok

# The campus microgrid: battery, library building and island frequency
E_KWH, ETA = 500.0, 0.95                 # battery: energy and efficiency of each direction
C_KWH_K, R_K_KW = 30.0, 0.4              # library: heat capacity and thermal resistance
M_KWS_HZ, D_KW_HZ = 80.0, 12.0           # island: inertia and damping of the loads
print("Setup complete. Running in Colab:", IN_COLAB, "| battery", E_KWH, "kWh | library time constant",
      C_KWH_K * R_K_KW, "h | island time constant", round(M_KWS_HZ / D_KW_HZ, 2), "s")

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 2.1": (lambda: len(np.arange(0, 60, 0.5)), 1e-9),
    "Exercise 2.2": (lambda: float(np.mean(np.linspace(10, 20, 11))), 1e-9),
    "Exercise 2.3": (lambda: 0.9 * 0.9, 1e-9),
    "Exercise A": (lambda: 45 - 20 * math.exp(-1), 1e-6),
    "Exercise B": (lambda: 0.8 - 150 * 2 / (0.95 * 500), 1e-6),
    "Exercise C": (lambda: -2 + 0.4 * 50, 1e-6),
    "Exercise D": (lambda: -44 / 80, 1e-6),
    "Exercise E": (lambda: float(STAGE2['heater start for 20 C at 8:00 (h after 20:00)']), 1e-6),
}

## Python step 2: Simulation loops with NumPy

A simulation advances a model in small steps of time. This step builds the tools: Arrays of time points, arithmetic on whole arrays, a loop that repeats one step of a model and a plot of the result &#91;[1](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lecture.html#ref-1), [2](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lecture.html#ref-2)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "D_KW_HZ" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-02/"
               "NB02_continuous_dynamics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### Arrays and time vectors

In [ ]:
# Python practice: a small example for learning the language
t = np.arange(0, 10, 2.5)              # start, stop (excluded), step
print("arange:", t)
u = np.linspace(0, 1, 5)               # start, stop (included), number of points
print("linspace:", u)
print("length:", len(t), "| last element:", t[-1], "| shape:", t.shape)

`np.arange` creates evenly spaced values up to, but not including, the stop value, and `np.linspace` a given number of values including both ends. An array of time points is the backbone of every simulation of this course.

### Arithmetic on whole arrays

In [ ]:
# Python practice: a small example for learning the language
hours = np.arange(0, 25, 6)
T = 5 + 16 * np.exp(-hours / 12)       # the library cooling down from 21 deg C
print(np.round(T, 2))
print("mean:", round(float(T.mean()), 2), "| below 12 deg C:", int((T < 12).sum()), "values")

Operations such as `*`, `+` and `np.exp` work element by element on whole arrays, so one line computes the temperature at all time points. A comparison such as `T < 12` gives an array of `True` and `False`, and `.sum()` counts the `True` values.

### The Euler step in a loop

In [ ]:
# Python practice: a small example for learning the language
dt, tau, x_ss = 0.5, 2.0, 10.0          # step, time constant, final value
x = np.zeros(9)
for k in range(8):
    x[k + 1] = x[k] + dt * (x_ss - x[k]) / tau   # one Euler step
print(np.round(x, 3))

The array `x` is created first and filled with zeros. The loop then fills it step by step: Each new value is the old value plus the step `dt` times the rate of change. Indexing with `x[k + 1]` writes the next element.

### Plots of trajectories

In [ ]:
# Python practice: a small example for learning the language
tt = np.arange(9) * dt
plt.figure(figsize=(5, 2.6))
plt.plot(tt, x, "o-", label="Euler, dt = 0.5")
plt.plot(tt, x_ss * (1 - np.exp(-tt / tau)), label="exact")
plt.xlabel("time")
plt.ylabel("x")
plt.legend()
plt.show()

`plt.plot(x, y, "o-")` draws points joined by lines, and `label` with `plt.legend()` names the curves. The plot shows the trajectory of the simulation next to the exact solution.

**Quick check.** Why does the loop above create `x` with nine elements for eight steps?

<details><summary>Answer</summary>

The first element holds the initial value; eight steps add eight more values.

</details>

### Exercises for Python step 2

**Exercise 2.1.** Create the time points from 0 to 60 s, without 60, in steps of 0.5 s with `np.arange`, and store their number in `n_points`.

In [ ]:
n_points = None   # your answer

check("Exercise 2.1", n_points)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_points = len(np.arange(0, 60, 0.5))

check("Exercise 2.1", n_points)

**Exercise 2.2.** Create 11 evenly spaced temperatures from 10 to 20 deg C with `np.linspace` and store their mean in `mean_T`.

In [ ]:
mean_T = None   # your answer

check("Exercise 2.2", mean_T)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
mean_T = float(np.mean(np.linspace(10, 20, 11)))

check("Exercise 2.2", mean_T)

**Exercise 2.3.** Start at `x = 1.0` and apply the step `x = x - 0.1 * x` twice in a loop. Store the result in `x2`.

In [ ]:
x2 = None   # your answer

check("Exercise 2.3", x2)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
x2 = 1.0
for _ in range(2):
    x2 = x2 - 0.1 * x2

check("Exercise 2.3", x2)

## 1. First-order systems and the Euler method &#91;[3](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lecture.html#ref-3)&#93;

*Lecture: [Differential equations: How physical quantities change](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lecture.html#differential-equations)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "tt" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-02/"
               "NB02_continuous_dynamics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Many physical quantities approach a final value at a rate proportional to the remaining distance. Python compares the exact solution of such a first-order system with the Euler method and shows how the step size decides between accuracy, error and instability.

**From the formulas to the code.** The exact solution $x(t) = x_{ss} + (x_0 - x_{ss})\,e^{-t/\tau}$ is `exact(t, x0, x_ss, tau)`. The Euler step $x[k+1] = x[k] + \Delta t\,(x_{ss} - x[k])/\tau$ is the body of `euler`, and the factor per step $1 - \Delta t/\tau$ is printed for several step sizes.

In [ ]:
def exact(t, x0, x_ss, tau):
    return x_ss + (x0 - x_ss) * np.exp(-np.asarray(t, float) / tau)

def euler(x0, x_ss, tau, dt, t_end):
    n = int(round(t_end / dt))
    x = np.empty(n + 1)
    x[0] = x0
    for k in range(n):
        x[k + 1] = x[k] + dt * (x_ss - x[k]) / tau
    return np.arange(n + 1) * dt, x

# the inverter cabinet warms from 25 to 45 deg C with a time constant of 20 min
print(f"exact after 20 min: {float(exact(20, 25, 45, 20)):.2f} C")
for dt in [1, 5, 10, 30, 45]:
    t, x = euler(25, 45, 20, dt, 90)
    i20 = np.argmin(np.abs(t - 20))
    print(f"dt = {dt:>2} min: factor 1 - dt/tau = {1 - dt / 20:5.2f}, "
          f"x at {t[i20]:.0f} min = {x[i20]:7.2f} C, x at 90 min = {x[-1]:7.2f} C")

**Python note.** `np.argmin(np.abs(t - 20))` finds the index of the time point closest to 20 min. With a step of 45 min, the Euler method jumps past the final value and oscillates, because the factor $1 - \Delta t/\tau$ is negative.

**Exercise A.** Store the exact temperature of the cabinet after 20 min, from 25 deg C toward 45 deg C with $\tau = 20$ min, in `T20`.

In [ ]:
T20 = None   # your answer

check("Exercise A", T20)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
T20 = float(exact(20, 25, 45, 20))

check("Exercise A", T20)

In [ ]:
# Figure: the exact solution and the Euler method for three step sizes
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
tt = np.linspace(0, 90, 300)
ax[0].plot(tt, exact(tt, 25, 45, 20), color="#1B2430", lw=2, label="exact")
for dt, c in [(5, "#1E6FA8"), (20, "#C98510"), (45, "#B53F2E")]:
    t, x = euler(25, 45, 20, dt, 90)
    ax[0].plot(t, x, "o-", color=c, ms=4, label=f"Euler, dt = {dt} min")
ax[0].set_xlabel("time (min)")
ax[0].set_ylabel("temperature (C)")
ax[0].legend(fontsize=8)
r = np.linspace(0, 2.5, 200)
ax[1].plot(r, np.abs(1 - r), color="#1E6FA8")
ax[1].axhline(1, ls=":", color="#B53F2E")
ax[1].set_xlabel("dt / tau")
ax[1].set_ylabel("|1 - dt/tau|")
plt.tight_layout()
plt.show()

**Observe and reflect.** Which step size would you choose for a simulation of a whole day of the library, whose time constant is 12 hours? Which for the frequency of the island, whose time constant is about 7 seconds?

## 2. The battery: State of charge and voltage &#91;[5](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lecture.html#ref-5)&#93;

*Lecture: [The battery: State of charge as an integrator](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lecture.html#the-battery)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "euler" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-02/"
               "NB02_continuous_dynamics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The state of charge of the battery integrates its power over time, with a loss in each direction. Python integrates the evening discharge of Week 1 minute by minute and computes the terminal voltage of the pack with a simple equivalent circuit.

**From the formulas to the code.** $\dot{SoC} = -P/(\eta E)$ for discharging and $-\eta P/E$ for charging is `d_soc(P)`. The open circuit voltage $V_{oc} = 700 + 120\,SoC$, the current $I = P/V_{oc}$ and the terminal voltage $V = V_{oc} - I R_0$ are computed in `terminal_voltage(soc, P)`.

In [ ]:
def d_soc(P_kw):
    """Rate of change of the state of charge per hour; P > 0 discharges, P < 0 charges."""
    return -P_kw / (ETA * E_KWH) if P_kw > 0 else -ETA * P_kw / E_KWH

def terminal_voltage(soc, P_kw, R0=0.04):
    v_oc = 700 + 120 * soc                     # a linear open-circuit voltage of the pack (V)
    current = P_kw * 1000 / v_oc               # A, positive when discharging
    return v_oc, current, v_oc - current * R0

def integrate_soc(soc0, powers_kw, dt_h):
    soc = [soc0]
    for P in powers_kw:
        soc.append(soc[-1] + dt_h * d_soc(P))
    return np.array(soc)

# The worked example: 200 kW for 1.5 h from 0.80
soc = integrate_soc(0.80, [200] * 90, 1 / 60)
print(f"after 1.5 h at 200 kW: SoC {soc[-1]:.4f} (formula {0.8 - 200 * 1.5 / (0.95 * 500):.4f})")
for s in [0.8, 0.5, 0.168]:
    v_oc, I, V = terminal_voltage(s, 200)
    print(f"SoC {s:.3f}: V_oc {v_oc:.1f} V, current {I:.1f} A, terminal {V:.1f} V")

**Python note.** `soc[-1]` is the last element of the list. Integrating with 90 steps of one minute gives exactly the result of the formula here, because the power is constant.

**Exercise B.** The battery discharges 150 kW for 2 hours from a state of charge of 0.8. Store the final state of charge in `soc_b`.

In [ ]:
soc_b = None   # your answer

check("Exercise B", soc_b)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
soc_b = 0.8 - 150 * 2 / (ETA * E_KWH)

check("Exercise B", soc_b)

In [ ]:
# Figure: the evening of Week 1, minute by minute
evening = [0] * 60 * 2 + [52.9] * 60 + [130.5] * 60 + [135.7] * 60 + [39.9] * 60   # 15:00 to 21:00, kW
soc_e = integrate_soc(0.9, evening, 1 / 60)
tt = 15 + np.arange(len(soc_e)) / 60
V = [terminal_voltage(s, P)[2] for s, P in zip(soc_e[:-1], evening)]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(tt, 100 * soc_e, color="#2E8B57")
ax[0].set_xlabel("hour")
ax[0].set_ylabel("state of charge (%)")
ax[1].plot(tt[:-1], V, color="#1E6FA8")
ax[1].set_xlabel("hour")
ax[1].set_ylabel("terminal voltage (V)")
plt.tight_layout()
plt.show()
print(f"state of charge at 21:00: {soc_e[-1]:.3f}")

**Observe and reflect.** Why does the terminal voltage jump down when the discharge starts, before the state of charge has changed much? What would a battery management system do if the voltage fell below a limit?

## 3. The library as a heat capacitor: Your nights &#91;[6](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lecture.html#ref-6)&#93;

*Lecture: [A building as a heat capacitor](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lecture.html#a-building-as-a-heat-capacitor)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "integrate_soc" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-02/"
               "NB02_continuous_dynamics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A building stores heat in its walls and air and loses it through its envelope. Python simulates the library with its resistance and capacity and finds when the heater must start so that the reading room is warm at opening time.

**From the formulas to the code.** The model $C\,\dot{T} = (T_{out} - T)/R + P$ is the body of `simulate_room`, with the time constant $\tau = R\,C$. The steady state $T_{ss} = T_{out} + R\,P$ and the exact solution are used in `time_to_reach(T0, T_target, P, T_out)`.

In [ ]:
def simulate_room(T0, P_kw, T_out, hours, dt_h=1 / 60):
    """Euler steps of C dT/dt = (T_out - T) / R + P; P may be a function of the time in hours."""
    n = int(round(hours / dt_h))
    T = np.empty(n + 1)
    T[0] = T0
    for k in range(n):
        p = P_kw(k * dt_h) if callable(P_kw) else P_kw
        T[k + 1] = T[k] + dt_h * ((T_out - T[k]) / R_K_KW + p) / C_KWH_K
    return np.arange(n + 1) * dt_h, T

def time_to_reach(T0, T_target, P_kw, T_out):
    T_ss = T_out + R_K_KW * P_kw
    return C_KWH_K * R_K_KW * math.log((T_ss - T0) / (T_ss - T_target))

tau = C_KWH_K * R_K_KW
print(f"time constant {tau:.0f} h | steady state with 40 kW at 5 C: {5 + R_K_KW * 40:.1f} C")
t, T = simulate_room(15, 40, 5, 3)
print(f"from 15 C with 40 kW after 3 h: {T[-1]:.2f} C (exact {21 - 6 * math.exp(-0.25):.2f} C)")
print(f"from 15 C to 20 C with 60 kW: {time_to_reach(15, 20, 60, 5):.2f} h")

**Python note.** `P_kw` may be a number or a function of the time, such as a schedule. `callable(P_kw)` tells which of the two it is. The Euler step of one minute is very small against the time constant of 12 h, so the simulation and the exact solution agree to two decimals.

**Exercise C.** Store the steady-state temperature of the library with a heater of 50 kW at an outdoor temperature of -2 deg C in `T_ss`.

In [ ]:
T_ss = None   # your answer

check("Exercise C", T_ss)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
T_ss = -2 + R_K_KW * 50

check("Exercise C", T_ss)

Part A of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lab.html) runs nights in the climate room of the library. Paste the copied rows into `MY_NIGHTS`, and the code compares each run with the model.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_NIGHTS = """
"""
if MY_NIGHTS.strip():
    nights = pd.read_csv(io.StringIO(MY_NIGHTS.strip()))
    source = "your own nights in the climate room"
else:
    nights = pd.DataFrame({"run": [1, 2, 3], "t_out": [5, 5, 5], "power_kw": [40, 60, 60],
                           "start_h": [0.0, 7.0, 5.5], "temp_at_8": [21.0, 19.1, 20.9]})
    source = "a simulated student"
model = []
for _, r in nights.iterrows():
    P = lambda t, r=r: r["power_kw"] if t >= r["start_h"] else 0.0
    t, T = simulate_room(21.0, P, r["t_out"], 12)
    model.append(round(float(T[-1]), 1))
nights["model_temp_at_8"] = model
print("Data:", source)
print(nights.to_string(index=False))

**Python note.** Each night starts at 20:00 with the room at 21 deg C, and `start_h` counts the hours after 20:00 until the heater starts. A start of 0 means that the heater runs all night.

In [ ]:
# Figure: three nights of the library
fig, ax = plt.subplots(figsize=(8, 3.4))
for label, P, c in [("heater off", 0.0, "#56616D"), ("40 kW all night", 40.0, "#2E8B57"),
                    ("60 kW from 1:00", lambda t: 60.0 if t >= 5 else 0.0, "#1E6FA8")]:
    t, T = simulate_room(21, P, 5, 12)
    ax.plot(20 + t, T, color=c, label=label)
ax.axhline(20, ls=":", color="#B53F2E")
ax.set_xlabel("hour (20 = 20:00, 32 = 8:00)")
ax.set_ylabel("temperature (C)")
ax.legend(fontsize=8)
plt.show()

**Observe and reflect.** Why does the room cool down slowly at first and then more and more slowly? What would a thicker insulation change in the model, $R$ or $C$?

## 4. The frequency of the island: Your load steps &#91;[7](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lecture.html#ref-7)&#93;

*Lecture: [Grid frequency: The swing of the island](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lecture.html#grid-frequency)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "simulate_room" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-02/"
               "NB02_continuous_dynamics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** In island mode, the rotating machines and the inverters with virtual inertia hold the frequency. Python simulates the deviation of the frequency after a load step, computes its initial rate of change and finds when an under-frequency relay would trip without control.

**From the formulas to the code.** $M\,\dot{\Delta f} = -D\,\Delta f - \Delta P$ is the body of `swing`. The rate of change $-\Delta P/M$, the final value $-\Delta P/D$ and the time constant $M/D$ are printed, with a droop gain $K$ added to $D$ when the battery helps.

In [ ]:
def swing(dP_kw, M=M_KWS_HZ, D=D_KW_HZ, K=0.0, T=20.0, dt=0.001, trip=2.5):
    """Frequency deviation after a load step dP at t = 0; returns time, deviation and the time of a trip."""
    n = int(T / dt)
    df = np.zeros(n + 1)
    t_trip = None
    for k in range(n):
        df[k + 1] = df[k] + dt * (-(D + K) * df[k] - dP_kw) / M
        if t_trip is None and abs(df[k + 1]) > trip:
            t_trip = (k + 1) * dt
    return np.arange(n + 1) * dt, df, t_trip

for K in [0.0, 125.0]:
    t, df, t_trip = swing(66, K=K)
    print(f"K = {K:>5}: RoCoF {(df[1] - df[0]) / 0.001:.3f} Hz/s, final {-66 / (D_KW_HZ + K):.3f} Hz, "
          f"time constant {M_KWS_HZ / (D_KW_HZ + K):.3f} s, trip at "
          f"{'no trip' if t_trip is None else f'{t_trip:.2f} s'}")
print(f"trip time by the formula: {-M_KWS_HZ / D_KW_HZ * math.log(1 - 2.5 / 5.5):.2f} s")

**Python note.** The first difference of the simulated deviation divided by the step gives the initial rate of change of frequency (RoCoF). The relay is modelled by the first time that the deviation exceeds 2.5 Hz.

**Exercise D.** Store the initial rate of change of frequency in Hz/s after a load step of 44 kW with $M = 80$ kW·s/Hz in `rocof`.

In [ ]:
rocof = None   # your answer

check("Exercise D", rocof)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
rocof = -44 / M_KWS_HZ

check("Exercise D", rocof)

Part C of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lab.html) switches load steps on a flywheel bench. Paste the copied rows into `MY_STEPS`, and the code compares your measured rate of change with the model.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_STEPS = """
"""
if MY_STEPS.strip():
    steps = pd.read_csv(io.StringIO(MY_STEPS.strip()))
    source = "your own load steps"
else:
    steps = pd.DataFrame({"trial": [1, 2, 3, 4], "inertia": [80, 80, 160, 80], "step_kw": [22, 66, 66, 66],
                          "droop": [0, 0, 0, 1], "rocof": [-0.27, -0.82, -0.41, -0.76]})
    source = "a simulated student"
steps["model_rocof"] = (-steps["step_kw"] / steps["inertia"]).round(3)
steps["model_final_hz"] = (-steps["step_kw"] / (D_KW_HZ + 125 * steps["droop"])).round(3)
print("Data:", source)
print(steps.to_string(index=False))

In [ ]:
# Figure: the deviation after 66 kW for two inertias, with and without the droop of the battery
fig, ax = plt.subplots(figsize=(8, 3.4))
for M, K, c, lab in [(80, 0, "#B53F2E", "M = 80, no droop"), (160, 0, "#C98510", "M = 160, no droop"),
                     (80, 125, "#2E8B57", "M = 80, droop K = 125")]:
    t, df, t_trip = swing(66, M=M, K=K, T=10)
    ax.plot(t, np.clip(df, -6, 1), color=c, label=lab + (f", trip at {t_trip:.2f} s" if t_trip else ""))
ax.axhline(-2.5, ls=":", color="#56616D")
ax.set_xlabel("time after the load step (s)")
ax.set_ylabel("frequency deviation (Hz)")
ax.legend(fontsize=8)
plt.show()

**Observe and reflect.** Why does a larger inertia slow the fall of the frequency but not change its final value without control? Which part of the model does the droop of the battery change? The meter of the interactive lab measures the slope over the first 0.1 s. Why does it read less than the model with the droop on?

## 5. The campus microgrid, stage 2

*Lecture: [The campus microgrid, stage 2: Dynamic models of the battery, the library and the frequency](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lecture.html#the-campus-microgrid-stage-2)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "swing" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-02/"
               "NB02_continuous_dynamics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 2 collects one result of each dynamic model. The first is how far the evening of Week 1 drains the battery. The second is when the heater of the library must start after a night setback. The third is what the largest load step does to the frequency of the island.

In [ ]:
STAGE2 = {}
STAGE2["state of charge at 21:00 after the evening shaving"] = round(float(soc_e[-1]), 3)
v_oc, I, V = terminal_voltage(soc_e[-2], evening[-1])
STAGE2["terminal voltage at the end of the shaving (V)"] = round(float(V), 1)

def night(start_h, P=60.0, T_out=5.0):
    """Heater off from 20:00, full power from start_h hours after 20:00; temperature at 8:00."""
    t, T = simulate_room(21, lambda tt: P if tt >= start_h else 0.0, T_out, 12)
    return float(T[-1])

lo, hi = 0.0, 12.0                 # bisection: the latest start that still reaches 20 C at 8:00
for _ in range(40):
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if night(mid) >= 20 else (lo, mid)
start = round(lo, 2)
STAGE2["heater start for 20 C at 8:00 (h after 20:00)"] = start
STAGE2["heating energy with the setback (kWh)"] = round(60 * (12 - lo), 1)
STAGE2["heating energy at 21 C all night (kWh)"] = round((21 - 5) / R_K_KW * 12, 1)
t, df, t_trip = swing(66)
STAGE2["RoCoF after 66 kW (Hz/s)"] = round(-66 / M_KWS_HZ, 3)
STAGE2["trip without control (s)"] = round(t_trip, 2)
STAGE2["final deviation with droop 125 kW/Hz (Hz)"] = round(-66 / (D_KW_HZ + 125), 3)
for key, value in STAGE2.items():
    print(f"{key:<52} {value}")

**Python note.** The latest start is found by bisection: The interval between a start that is early enough and one that is too late is halved forty times. The function `night` returns the temperature at 8:00 for a start time.

In [ ]:
# Figure: the night of the library with the latest start, and the frequency after 66 kW
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
t, T = simulate_room(21, lambda tt: 60.0 if tt >= start else 0.0, 5, 12)
ax[0].plot(20 + t, T, color="#1E6FA8")
ax[0].axvline(20 + start, ls=":", color="#C98510")
ax[0].axhline(20, ls=":", color="#B53F2E")
ax[0].set_xlabel("hour (32 = 8:00)")
ax[0].set_ylabel("temperature (C)")
for K, c in [(0, "#B53F2E"), (125, "#2E8B57")]:
    t, df, _ = swing(66, K=K, T=8)
    ax[1].plot(t, np.clip(df, -6, 1), color=c, label=f"K = {K}")
ax[1].axhline(-2.5, ls=":", color="#56616D")
ax[1].set_xlabel("time (s)")
ax[1].set_ylabel("deviation (Hz)")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Exercise E.** Store the latest start of the heater, in hours after 20:00, as kept in `STAGE2`, in `start_e`.

In [ ]:
start_e = None   # your answer

check("Exercise E", start_e)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
start_e = STAGE2["heater start for 20 C at 8:00 (h after 20:00)"]

check("Exercise E", start_e)

**Observe and reflect.** How much heating energy does the setback save in this night, and what does it cost the users of the library? Why must the island never meet a load step of 66 kW without the droop of the battery?

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE2" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-02/"
               "NB02_continuous_dynamics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** First-order models describe many systems: A water tank, an electric car, a server room and a cold store. The functions of this week simulate them after one change of a name.

The systems and their numbers are illustrative. Set `APP` to one of the six names and run the cell.

In [ ]:
APP = "server_room"
# choose: library, water_tank, ev_battery, server_room, cold_store, greenhouse
APPS = {  # (description, x0, x_ss, time constant in h, unit, hours to simulate)
    "library": ("reading room heated with 60 kW at 5 C outside", 15.0, 29.0, 12.0, "C", 24),
    "water_tank": ("tank filling to the level of its float valve", 0.4, 2.0, 0.8, "m", 4),
    "ev_battery": ("car battery charging in its constant-voltage phase", 0.80, 1.0, 0.5, "SoC", 2),
    "server_room": ("server room after a failure of the cooling", 22.0, 48.0, 0.25, "C", 1),
    "cold_store": ("cold store after the door is closed", 9.0, -18.0, 3.0, "C", 12),
    "greenhouse": ("greenhouse heated in the morning sun", 8.0, 27.0, 1.5, "C", 6)}
desc, x0, x_ss, tau_h, unit, hours = APPS[APP]
t = np.linspace(0, hours, 7)
print(f"{APP}: {desc}")
for tt, xx in zip(t, exact(t, x0, x_ss, tau_h)):
    print(f"  t = {tt:5.2f} h: {xx:7.2f} {unit}")
print(f"  63 percent of the way after one time constant: {x0 + 0.632 * (x_ss - x0):.2f} {unit} at {tau_h} h")

**Python note.** Each entry holds the initial value, the final value and the time constant of a first-order system. The exact solution of Section 1 gives the trajectory for all of them.

**Observe and reflect.** Which of the systems would need a controller that acts within minutes, and which within hours? What does the time constant tell you about the sampling period of its sensor?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Write one function `simulate(f, x0, dt, t_end)` that takes the right-hand side of any first-order model as a function, and use it for the battery, the room and the frequency. |
| Embedded systems | A microcontroller computes the Euler step of the room model once a minute with 16-bit integers in tenths of a degree. Simulate the rounding and compare with the float result. |
| Internet of Things | Simulate a smart thermostat that lowers the setpoint at night and starts preheating at the latest time found in stage 2 for each outdoor temperature from -10 to 10 deg C. |
| Computer networks | The measurement of the frequency arrives 0.2 s late. Add the delay to `swing` with a droop of 125 kW/Hz and compare the lowest frequency. |
| Control engineering | Replace the Euler method by the method of Heun or the fourth-order Runge-Kutta method and compare the error for a step of 10 min on the cabinet example. |
| Data science | Fit the time constant of the room from a simulated cooling curve with noise by a straight line through the logarithm of the temperature difference. |
| Cybersecurity | An attacker changes the parameter $R$ in the controller of the library by 50 percent. How late or early would the heater start, and how would you detect it? |
| Artificial intelligence | Train a small regression model on simulated nights to predict the start time of the heater from the outdoor temperature, and compare it with the formula. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-02).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>